In [ ]:

def build_features(df_input, spy_input):
    """
    Master feature engineering function.
    Takes raw OHLCV data and returns a rich feature DataFrame.
    Every feature is documented with its financial intuition.
    """
    df = df_input.copy()
    spy = spy_input.copy()

    # ── RETURNS (the foundation) ─────────────────────────────────
    df['return_1d']  = df['Close'].pct_change(1)   # 1-day return
    df['return_3d']  = df['Close'].pct_change(3)   # 3-day return
    df['return_5d']  = df['Close'].pct_change(5)   # 1-week return
    df['return_10d'] = df['Close'].pct_change(10)  # 2-week return
    df['return_20d'] = df['Close'].pct_change(20)  # 1-month return

    # Lagged returns: what happened in the recent past?
    # These capture short-term momentum and mean-reversion signals
    for lag in [1, 2, 3, 5]:
        df[f'return_lag{lag}'] = df['return_1d'].shift(lag)



In [ ]:
 # ── TREND INDICATORS ─────────────────────────────────────────
    # Simple Moving Averages
    for window in [5, 10, 20, 50, 200]:
        df[f'SMA_{window}'] = df['Close'].rolling(window).mean()

    # SMA ratios: where is price relative to its moving averages?
    df['price_vs_SMA10']  = df['Close'] / df['SMA_10']  - 1
    df['price_vs_SMA50']  = df['Close'] / df['SMA_50']  - 1
    df['price_vs_SMA200'] = df['Close'] / df['SMA_200'] - 1
    df['SMA10_vs_SMA50']  = df['SMA_10'] / df['SMA_50'] - 1

    # EMA (Exponential Moving Average): weights recent data more heavily
    df['EMA_12'] = df['Close'].ewm(span=12, adjust=False).mean()
    df['EMA_26'] = df['Close'].ewm(span=26, adjust=False).mean()

    # MACD: difference between fast and slow EMA
    # Positive MACD = short-term trend above long-term = bullish momentum
    df['MACD']        = df['EMA_12'] - df['EMA_26']
    df['MACD_signal'] = df['MACD'].ewm(span=9, adjust=False).mean()
    df['MACD_hist']   = df['MACD'] - df['MACD_signal']  # Histogram: momentum of momentum


In [ ]:
   # ── MOMENTUM INDICATORS ──────────────────────────────────────
    # RSI (14-day): measures speed and magnitude of price changes
    def compute_rsi(series, period=14):
        delta    = series.diff()
        gain     = delta.clip(lower=0).rolling(period).mean()
        loss     = (-delta.clip(upper=0)).rolling(period).mean()
        rs       = gain / loss.replace(0, np.nan)
        return 100 - (100 / (1 + rs))

    df['RSI_14'] = compute_rsi(df['Close'], 14)
    df['RSI_7']  = compute_rsi(df['Close'], 7)   # Short-term RSI

    # RSI divergence: is RSI's trend different from price's trend?
    # Divergence often precedes reversals
    df['RSI_14_change'] = df['RSI_14'].diff()
    df['RSI_vs_50']     = df['RSI_14'] - 50  # Centered at 50 (neutral)

    # Rate of Change (ROC): pure momentum
    df['ROC_5']  = df['Close'].pct_change(5)  * 100
    df['ROC_20'] = df['Close'].pct_change(20) * 100



In [ ]:
# ── VOLATILITY INDICATORS ─────────────────────────────────────
    # Historical Volatility (annualized)
    df['vol_5d']  = df['return_1d'].rolling(5).std()  * np.sqrt(252)
    df['vol_20d'] = df['return_1d'].rolling(20).std() * np.sqrt(252)
    df['vol_60d'] = df['return_1d'].rolling(60).std() * np.sqrt(252)

    # Volatility ratio: is current volatility high or low relative to recent past?
    # Used to detect volatility breakouts and compressions
    df['vol_ratio'] = df['vol_5d'] / df['vol_20d']

    # Bollinger Bands
    bb_mid = df['Close'].rolling(20).mean()
    bb_std = df['Close'].rolling(20).std()
    df['BB_upper']    = bb_mid + 2 * bb_std
    df['BB_lower']    = bb_mid - 2 * bb_std
    df['BB_width']    = (df['BB_upper'] - df['BB_lower']) / bb_mid  # Normalized band width
    df['BB_position'] = (df['Close'] - df['BB_lower']) / (df['BB_upper'] - df['BB_lower'])
    # Position 0 = at lower band, 0.5 = at midpoint, 1 = at upper band

    # Average True Range (ATR): measures daily price range / volatility
    df['TR'] = np.maximum(
        df['High'] - df['Low'],
        np.maximum(
            abs(df['High'] - df['Close'].shift(1)),
            abs(df['Low']  - df['Close'].shift(1))
        )
    )
    df['ATR_14'] = df['TR'].rolling(14).mean()
    df['ATR_ratio'] = df['ATR_14'] / df['Close']  # Normalized ATR


In [ ]:
 # ── VOLUME INDICATORS ─────────────────────────────────────────
    # Volume ratio: is today's volume above the recent average?
    df['vol_ratio_20'] = df['Volume'] / df['Volume'].rolling(20).mean()

    # On-Balance Volume (OBV): running total of volume, direction by price movement
    # Rising OBV with flat price = accumulation (bullish)
    obv = [0]
    for i in range(1, len(df)):
        if df['Close'].iloc[i] > df['Close'].iloc[i-1]:
            obv.append(obv[-1] + df['Volume'].iloc[i])
        elif df['Close'].iloc[i] < df['Close'].iloc[i-1]:
            obv.append(obv[-1] - df['Volume'].iloc[i])
        else:
            obv.append(obv[-1])
    df['OBV'] = obv
    df['OBV_change'] = df['OBV'].pct_change(5)  # 5-day OBV change

    # Volume-weighted price: is price moving with or against volume?
    df['VWAP_ratio'] = df['Close'] / ((df['Close'] * df['Volume']).rolling(20).sum() /
                                        df['Volume'].rolling(20).sum())



In [ ]:
# ── INTRADAY FEATURES ─────────────────────────────────────────
    # These capture the relationship between open, high, low, close
    df['hl_range']    = (df['High'] - df['Low'])  / df['Close']   # Daily range (normalized)
    df['oc_range']    = (df['Close'] - df['Open']) / df['Open']   # Open-to-close return
    df['upper_wick']  = (df['High'] - np.maximum(df['Open'], df['Close'])) / df['Close']
    df['lower_wick']  = (np.minimum(df['Open'], df['Close']) - df['Low']) / df['Close']
    # Large lower wicks = buyers stepping in at lows (bullish signal)


In [ ]:
 # ── MARKET FEATURES (SPY as market proxy) ─────────────────────
    spy['spy_return_1d'] = spy['Close'].pct_change(1)
    spy['spy_return_5d'] = spy['Close'].pct_change(5)

    # Align SPY data to our stock's index
    df['spy_return_1d'] = spy['spy_return_1d'].reindex(df.index)
    df['spy_return_5d'] = spy['spy_return_5d'].reindex(df.index)

    # Relative strength vs market: is AAPL outperforming the S&P?
    df['rel_strength_5d']  = df['return_5d']  - df['spy_return_5d']



In [ ]:

    # ── CALENDAR FEATURES ─────────────────────────────────────────
    # These encode time-of-year effects (seasonal patterns)
    df['day_of_week']     = df.index.dayofweek         # 0=Mon, 4=Fri
    df['month']           = df.index.month
    df['quarter']         = df.index.quarter
    df['is_month_end']    = df.index.is_month_end.astype(int)
    df['is_quarter_end']  = df.index.is_quarter_end.astype(int)

    return df


# Run the feature builder
df_features = build_features(raw_data["AAPL"], raw_data["SPY"])
print(f"Feature set: {df_features.shape[1]} columns, {df_features.shape[0]} rows")
python
# --- Feature correlation with the target ---
# Before modeling, check which features are actually predictive.
# Weak correlation doesn't mean useless — but strong correlation is promising.

feature_cols = [
    'return_lag1', 'return_lag2', 'return_lag3', 'return_lag5',
    'return_5d', 'return_10d', 'return_20d',
    'RSI_14', 'RSI_7', 'RSI_vs_50',
    'MACD', 'MACD_hist',
    'BB_position', 'BB_width',
    'vol_ratio', 'ATR_ratio',
    'vol_ratio_20', 'OBV_change', 'VWAP_ratio',
    'price_vs_SMA10', 'price_vs_SMA50', 'SMA10_vs_SMA50',
    'hl_range', 'oc_range', 'lower_wick', 'upper_wick',
    'spy_return_1d', 'rel_strength_5d',
    'day_of_week', 'month'
]

df_features['Target'] = (df_features['Close'].shift(-1) > df_features['Close']).astype(int)

# Compute point-biserial correlation of each feature with the binary target
correlations = {}
for col in feature_cols:
    clean = df_features[[col, 'Target']].dropna()
    corr, pval = stats.pointbiserialr(clean[col], clean['Target'])
    correlations[col] = {'correlation': corr, 'p_value': pval, 'significant': pval < 0.05}

corr_df = pd.DataFrame(correlations).T.sort_values('correlation', key=abs, ascending=False)
print("\nTop 15 features by correlation with target:")
print(corr_df.head(15).round(4))

# Visualize
top_features = corr_df.head(20)
colors = ['steelblue' if c > 0 else 'salmon' for c in top_features['correlation']]
plt.figure(figsize=(10, 7))
plt.barh(top_features.index, top_features['correlation'], color=colors)
plt.axvline(0, color='black', linewidth=0.8)
plt.title('Feature-Target Correlation (blue=bullish, red=bearish signal)')
plt.xlabel('Point-biserial correlation')
plt.tight_layout()
plt.show()